In [90]:
import os
import pandas as pd
import numpy as np
import matplotlib
import sklearn
print(os.getcwd())

/home/michele/big-project-FOR


In [91]:
edificio1= pd.read_csv('Edificio1.csv')
edificio2= pd.read_csv('Edificio2.csv')

edificio1_pairs= edificio1.rename_axis('index1').reset_index().merge(edificio1.rename_axis('index2').reset_index(), how='cross')
edificio2_pairs= edificio2.rename_axis('index1').reset_index().merge(edificio2.rename_axis('index2').reset_index(), how='cross')

In [92]:
first_point_cols= ['x_x','y_x','z_x']
second_point_cols=['x_y','y_y','z_y']
diff_cols= ['dx','dy','dz']

In [93]:
def process_edificio(dataframe):
    #distances and deltas on coordinates
    dataframe[diff_cols] = (dataframe[first_point_cols].values - dataframe[second_point_cols].values)
    dataframe['euclidean_distance'] = np.sqrt((dataframe[diff_cols]**2).sum(axis=1)) #axis=1 for row-wise sum
    
    #absolute values on deltas
    dataframe[diff_cols] = dataframe[diff_cols].abs()
    
    # conditions
    condition_near = (dataframe['euclidean_distance'] <= 4.0)
    
    
    condition_tunnel = (
        ((dataframe['dx'] <= 0.5) & (dataframe['dy'] <= 0.5)) | 
        ((dataframe['dy'] <= 0.5) & (dataframe['dz'] <= 0.5)) | 
        ((dataframe['dx'] <= 0.5) & (dataframe['dz'] <= 0.5))
    )
    condition_tunnel_and_not_far = (dataframe['euclidean_distance'] <= 11.0) & condition_tunnel
    
    # final connection condition
    dataframe["are_connected"] = condition_near | condition_tunnel_and_not_far
    return dataframe



In [94]:
edificio1_pairs = process_edificio(edificio1_pairs)
edificio2_pairs = process_edificio(edificio2_pairs)

print(edificio1_pairs.head())
print("\nSHAPE", edificio1_pairs.shape)

   index1  x_x   y_x  z_x  index2  x_y    y_y   z_y   dx   dy    dz  \
0       0  2.5  3.25  0.0       0  2.5   3.25   0.0  0.0  0.0   0.0   
1       0  2.5  3.25  0.0       1 -6.0   3.25   0.0  8.5  0.0   0.0   
2       0  2.5  3.25  0.0       2 -6.0  11.75   0.0  8.5  8.5   0.0   
3       0  2.5  3.25  0.0       3  2.5  11.75   0.0  0.0  8.5   0.0   
4       0  2.5  3.25  0.0       4  2.5   3.25  11.0  0.0  0.0  11.0   

   euclidean_distance  are_connected  
0            0.000000           True  
1            8.500000           True  
2           12.020815          False  
3            8.500000           True  
4           11.000000           True  

SHAPE (4096, 13)
